# Intro to CFR

Working through the paper **"An Introduction to Counterfactual Regret Minimization"** by Todd W. Neller & Marc Lanctot.

The notebook reproduces the code examples, solves the exercises, and builds intuition for CFR step by step — from regret matching in simple matrix games to full CFR on Kuhn Poker.

### Worked Example 1: Rock-Paper-Scissors (Section 2.4)

Regret matching for a single player computing a best response against a fixed opponent strategy.

Reference: *An Introduction to Counterfactual Regret Minimization*, Neller & Lanctot, Section 2.  
(original code in the paper is in java, traduced here in Python)

In [4]:
#Imports 

import random
import numpy as np

In [5]:
class RPSTrainer:
    def __init__(self, opp_strategy=np.array([0.4, 0.3, 0.3])):
        self.NUM_ACTIONS = 3
        self.regret_sum = np.zeros(self.NUM_ACTIONS)
        self.strategy = np.zeros(self.NUM_ACTIONS)
        self.strategy_sum = np.zeros(self.NUM_ACTIONS)
        self.opp_strategy = opp_strategy

    def get_strategy(self):
        self.strategy = np.maximum(self.regret_sum, 0)
        normalizing_sum = self.strategy.sum()

        if normalizing_sum > 0:
            self.strategy /= normalizing_sum
        else:
            self.strategy = np.ones(self.NUM_ACTIONS) / self.NUM_ACTIONS

        self.strategy_sum += self.strategy
        return self.strategy

    def get_action(self, strategy):
        return np.random.choice(self.NUM_ACTIONS, p=strategy)

    def train(self, iterations):
        for i in range(iterations):
            strategy = self.get_strategy()
            my_action = self.get_action(strategy)
            opp_action = self.get_action(self.opp_strategy)

            action_utility = np.zeros(self.NUM_ACTIONS)
            action_utility[opp_action] = 0
            action_utility[(opp_action + 1) % self.NUM_ACTIONS] = 1
            action_utility[(opp_action - 1) % self.NUM_ACTIONS] = -1

            self.regret_sum += action_utility - action_utility[my_action]

    def get_average_strategy(self):
        normalizing_sum = self.strategy_sum.sum()
        if normalizing_sum > 0:
            return self.strategy_sum / normalizing_sum
        else:
            return np.ones(self.NUM_ACTIONS) / self.NUM_ACTIONS


**Regret Matching** (Section 2.4) — L'idée centrale :

1. À chaque itération, on calcule le **regret** de ne pas avoir joué chaque action (= utilité alternative − utilité de l'action jouée).
2. On accumule ces regrets au fil du temps.
3. La stratégie suivante est proportionnelle aux **regrets positifs** cumulés.

Ici, l'adversaire joue une stratégie **fixe** (0.4 R, 0.3 P, 0.3 S). Les utilités espérées sont R=0, P=+0.1, S=−0.1 → la **stratégie moyenne** converge vers Papier pur (best response).

In [ ]:
# Run
trainer = RPSTrainer()
trainer.train(1000)
print(trainer.get_average_strategy())

[1.16666667e-03 9.98500000e-01 3.33333333e-04]


### Exercise 2.5 : Self-Play — Convergence vers l'équilibre de Nash

On modifie le programme pour que **les deux joueurs** utilisent le regret matching simultanément. Chacun maintient sa propre table de regrets et met à jour sa stratégie indépendamment.

**Résultat clé** : dans tout jeu à somme nulle à deux joueurs, quand les deux joueurs minimisent leur regret, la paire de **stratégies moyennes** converge vers un **équilibre de Nash**. Pour RPS, l'unique équilibre est (1/3, 1/3, 1/3).

In [6]:
class RPSTrainerSelfPlay:
    def __init__(self):
        self.NUM_ACTIONS = 3
        self.regret_sum = [np.zeros(3), np.zeros(3)]
        self.strategy_sum = [np.zeros(3), np.zeros(3)]

    def get_strategy(self, player):
        strategy = np.maximum(self.regret_sum[player], 0)
        normalizing_sum = strategy.sum()

        if normalizing_sum > 0:
            strategy /= normalizing_sum
        else:
            strategy = np.ones(self.NUM_ACTIONS) / self.NUM_ACTIONS

        self.strategy_sum[player] += strategy
        return strategy

    def get_action(self, strategy):
        return np.random.choice(self.NUM_ACTIONS, p=strategy)

    def train(self, iterations):
        for i in range(iterations):
            strategies = [self.get_strategy(0), self.get_strategy(1)]
            actions = [self.get_action(strategies[0]), self.get_action(strategies[1])]

            for player in range(2):
                opp_action = actions[1 - player]
                action_utility = np.zeros(self.NUM_ACTIONS)
                action_utility[opp_action] = 0
                action_utility[(opp_action + 1) % self.NUM_ACTIONS] = 1
                action_utility[(opp_action - 1) % self.NUM_ACTIONS] = -1

                self.regret_sum[player] += action_utility - action_utility[actions[player]]

    def get_average_strategy(self, player):
        normalizing_sum = self.strategy_sum[player].sum()
        if normalizing_sum > 0:
            return self.strategy_sum[player] / normalizing_sum
        else:
            return np.ones(self.NUM_ACTIONS) / self.NUM_ACTIONS



In [39]:
# Run
trainer = RPSTrainerSelfPlay()
trainer.train(1000000)
print("Player 1:", trainer.get_average_strategy(0))
print("Player 2:", trainer.get_average_strategy(1))

Player 1: [0.33265905 0.33360389 0.33373706]
Player 2: [0.33338102 0.33377953 0.33283945]


### Exercise 2.6 : Colonel Blotto (S=5, N=3)

Deux commandants répartissent **S=5 soldats** sur **N=3 champs de bataille**. Celui qui en remporte le plus gagne (+1), sinon match nul (0) ou défaite (−1).

**Approche :**
1. Énumérer toutes les **stratégies pures** = partitions de 5 en 3 parts ≥ 0 (stars & bars → 21 stratégies).
2. Définir `utility(s1, s2)` qui compare champ par champ.
3. Self-play avec regret matching sur ces 21 actions → convergence vers l'équilibre de Nash.

In [7]:
class BlottoTrainer:
    def __init__(self, S=5, N=3):
        # Enumerate all pure strategies (partitions of S into N non-negative parts)
        self.strategies = []
        for a in range(S + 1):
            for b in range(S + 1 - a):
                self.strategies.append((a, b, S - a - b))

        self.NUM_ACTIONS = len(self.strategies)
        self.regret_sum = [np.zeros(self.NUM_ACTIONS), np.zeros(self.NUM_ACTIONS)]
        self.strategy_sum = [np.zeros(self.NUM_ACTIONS), np.zeros(self.NUM_ACTIONS)]

    @staticmethod
    def utility(s1, s2):
        """Compare battlefield by battlefield. +1 win, -1 loss, 0 draw."""
        wins = sum(a > b for a, b in zip(s1, s2))
        losses = sum(a < b for a, b in zip(s1, s2))
        return (wins > losses) - (wins < losses)

    def get_strategy(self, player):
        strat = np.maximum(self.regret_sum[player], 0)
        total = strat.sum()
        if total > 0:
            strat /= total
        else:
            strat = np.ones(self.NUM_ACTIONS) / self.NUM_ACTIONS
        self.strategy_sum[player] += strat
        return strat

    def get_action(self, strategy):
        return np.random.choice(self.NUM_ACTIONS, p=strategy)

    def train(self, iterations):
        for _ in range(iterations):
            strats = [self.get_strategy(0), self.get_strategy(1)]
            actions = [self.get_action(strats[0]), self.get_action(strats[1])]

            for player in range(2):
                opp_action = actions[1 - player]
                action_utilities = np.array([
                    self.utility(self.strategies[a], self.strategies[opp_action])
                    for a in range(self.NUM_ACTIONS)
                ])
                self.regret_sum[player] += action_utilities - action_utilities[actions[player]]

    def get_average_strategy(self, player):
        total = self.strategy_sum[player].sum()
        if total > 0:
            return self.strategy_sum[player] / total
        return np.ones(self.NUM_ACTIONS) / self.NUM_ACTIONS

    def print_equilibrium(self, threshold=0.01):
        for player in range(2):
            avg = self.get_average_strategy(player)
            print(f"Player {player + 1} equilibrium strategy:")
            for i, prob in enumerate(avg):
                if prob > threshold:
                    print(f"  {self.strategies[i]} : {prob:.3f}")
            print()

In [22]:
# Run
trainer = BlottoTrainer(S=5, N=3)
print(f"{trainer.NUM_ACTIONS} pure strategies")
trainer.train(1_000)
trainer.print_equilibrium()

21 pure strategies
Player 1 equilibrium strategy:
  (0, 2, 3) : 0.137
  (0, 3, 2) : 0.157
  (1, 1, 3) : 0.086
  (1, 3, 1) : 0.088
  (2, 0, 3) : 0.087
  (2, 3, 0) : 0.128
  (3, 0, 2) : 0.116
  (3, 1, 1) : 0.095
  (3, 2, 0) : 0.084

Player 2 equilibrium strategy:
  (0, 2, 3) : 0.098
  (0, 3, 2) : 0.105
  (1, 1, 3) : 0.113
  (1, 3, 1) : 0.092
  (2, 0, 3) : 0.127
  (2, 3, 0) : 0.142
  (3, 0, 2) : 0.111
  (3, 1, 1) : 0.097
  (3, 2, 0) : 0.096



### Vérification par LP (programmation linéaire)

On résout le jeu **exactement** via le théorème minimax. Le joueur 1 cherche à maximiser son gain garanti $v$ :

$$\max \; v \quad \text{s.t.} \quad \sum_i x_i \cdot A_{ij} \geq v \;\; \forall j, \quad \sum_i x_i = 1, \quad x_i \geq 0$$

Cela permet de comparer avec la solution approximée par regret matching.

In [23]:
from scipy.optimize import linprog

# Build payoff matrix A[i,j] = utility of strategy i vs strategy j
strategies = trainer.strategies
n = len(strategies)
A = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        A[i, j] = BlottoTrainer.utility(strategies[i], strategies[j])

# LP: max v  s.t.  A^T x >= v,  sum(x) = 1,  x >= 0
# Rewrite as: min -v  s.t.  -A^T x + v <= 0,  sum(x) = 1,  x >= 0
# Variables: [x_0, ..., x_{n-1}, v]

# Objective: minimize -v (last variable)
c = np.zeros(n + 1)
c[-1] = -1

# Inequality constraints: -A^T x + v*1 <= 0  (one per opponent pure strategy j)
A_ub = np.zeros((n, n + 1))
A_ub[:, :n] = -A.T
A_ub[:, -1] = 1
b_ub = np.zeros(n)

# Equality constraint: sum(x) = 1
A_eq = np.zeros((1, n + 1))
A_eq[0, :n] = 1
b_eq = np.array([1.0])

# Bounds: x_i >= 0, v unbounded
bounds = [(0, None)] * n + [(None, None)]

result = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds)

x_eq = result.x[:n]
v_eq = result.x[-1]

print(f"Game value: {v_eq:.4f}")
print(f"\nNash equilibrium (exact LP):")
for i, prob in enumerate(x_eq):
    if prob > 0.001:
        print(f"  {strategies[i]} : {prob:.4f}")

print(f"\nRegret matching comparison:")
avg = trainer.get_average_strategy(0)
for i, prob in enumerate(avg):
    if prob > 0.01:
        print(f"  {strategies[i]} : {prob:.4f}")

Game value: -0.0000

Nash equilibrium (exact LP):
  (1, 1, 3) : 0.2500
  (1, 3, 1) : 0.2500
  (3, 0, 2) : 0.2500
  (3, 2, 0) : 0.2500

Regret matching comparison:
  (0, 2, 3) : 0.1372
  (0, 3, 2) : 0.1568
  (1, 1, 3) : 0.0864
  (1, 3, 1) : 0.0877
  (2, 0, 3) : 0.0868
  (2, 3, 0) : 0.1275
  (3, 0, 2) : 0.1160
  (3, 1, 1) : 0.0954
  (3, 2, 0) : 0.0836


In [26]:
# Cross-play: LP strategy vs Regret Matching strategy
lp_strat = x_eq
rm_strat = trainer.get_average_strategy(1)

# Compute expected payoff: E[u] = lp^T * A * rm
# LP as player 1, RM as player 2
payoff_lp_vs_rm = lp_strat @ A @ rm_strat
print(f"LP (P1) vs RM (P2): {payoff_lp_vs_rm:.12f}")

LP (P1) vs RM (P2): 0.001751773567
